# Lanjutan: Deteksi Wajah Asli vs AI (Sesi Colab Baru)

Notebook ini **melanjutkan** proyek CIFAKE sebelumnya, untuk mengatasi
temuan bahwa model CIFAKE tidak bisa menggeneralisasi ke foto wajah/selfie
manusia (karena CIFAR-10 tidak punya kelas manusia sama sekali).

**Yang beda dari notebook sebelumnya:**
1. Dataset: **140k Real and Fake Faces** (xhlulu, Kaggle) — 70.000 wajah
   asli dari FFHQ (Flickr-Faces-HQ, Nvidia) + 70.000 wajah buatan StyleGAN.
2. **Warm-start / transfer learning**: model CIFAKE yang sudah kamu latih
   (`cifake_cnn_best.keras`) di-upload lagi di sini dan dipakai sebagai
   titik awal bobot, bukan training dari nol. Arsitekturnya memakai
   `GlobalAveragePooling2D` sehingga bobot lama tetap kompatibel walau
   resolusi input berubah dari 32×32 ke 128×128 (ini sudah saya
   verifikasi langsung, tidak ada mismatch shape).
3. Resolusi gambar dinaikkan ke **128×128** (dari 32×32) supaya detail
   wajah/artefak GAN tidak hilang saat resize.

**Referensi dataset:**
- 140k Real and Fake Faces (xhlulu, Kaggle):
  https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
- Real: Flickr-Faces-HQ (FFHQ) — Karras, T., Laine, S., & Aila, T. (2019).
  *A Style-Based Generator Architecture for Generative Adversarial Networks*. CVPR.
- Fake: StyleGAN-generated faces, sampel dari 1 juta gambar yang dipublikasikan Bojan.

---
### Langkah 0 — Aktifkan GPU
**Runtime > Change runtime type > Hardware accelerator > GPU (T4)** > Save.


In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("✅ GPU terdeteksi:", gpus) if gpus else print("⚠️ Tidak ada GPU, cek Runtime > Change runtime type.")


---
### Langkah 1 — Install dependency

In [ ]:
!pip install -q kaggle scikit-learn pillow matplotlib
print("Selesai.")


---
### Langkah 2 — Tulis ulang source code proyek (sudah termasuk mekanisme warm-start)

In [ ]:
import os
for d in ["src", "app", "data", "models", "reports"]:
    os.makedirs(d, exist_ok=True)
print("Struktur folder dibuat:", os.listdir("."))


In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/config.py
"""
config.py
Semua konstanta & hyperparameter proyek ada di sini.

Proyek ini adalah LANJUTAN dari cifake-detector: masalah yang ditemukan
di versi sebelumnya (model CIFAKE gagal generalisasi ke foto wajah/selfie
manusia, karena CIFAR-10 tidak punya kelas manusia sama sekali) diatasi
dengan retraining di dataset wajah asli vs AI, memakai bobot model CIFAKE
sebagai titik awal (transfer learning), bukan dari nol.
"""

import os

# ---- Path dataset ----
# Dataset: "140k Real and Fake Faces" (xhlulu, Kaggle)
# https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
# 70.000 wajah asli dari FFHQ (Flickr-Faces-HQ, Nvidia) + 70.000 wajah
# buatan StyleGAN. Sudah dibagi train/valid/test oleh pembuat dataset.
#
# Struktur folder yang diharapkan setelah download & extract (biasanya ada
# folder pembungkus "real_vs_fake/real-vs-fake/" -- notebook Colab akan
# otomatis mendeteksi & merapikannya, jadi tidak perlu diubah manual):
#
# data/
#   train/
#     real/
#     fake/
#   valid/
#     real/
#     fake/
#   test/
#     real/
#     fake/

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VALID_DIR = os.path.join(DATA_DIR, "valid")
TEST_DIR = os.path.join(DATA_DIR, "test")

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

# Model CIFAKE lama (dilatih di tugas sebelumnya) dipakai sebagai warm-start.
# Upload file ini saat diminta notebook Colab, atau taruh manual di path ini
# kalau jalan di lokal/VS Code.
PRETRAINED_CIFAKE_PATH = os.path.join(MODELS_DIR, "cifake_cnn_best.keras")

KERAS_MODEL_PATH = os.path.join(MODELS_DIR, "face_cnn.keras")
BEST_CHECKPOINT_PATH = os.path.join(MODELS_DIR, "face_cnn_best.keras")
TFLITE_MODEL_PATH = os.path.join(MODELS_DIR, "face_cnn.tflite")
TFLITE_QUANT_MODEL_PATH = os.path.join(MODELS_DIR, "face_cnn_quant.tflite")

# ---- Gambar & data ----
# Wajah butuh resolusi lebih tinggi dari CIFAKE (32x32) supaya artefak
# GAN/diffusion yang halus tidak hilang saat resize. 128x128 dipilih
# sebagai titik tengah: cukup detail, masih ringan & cepat di GPU Colab.
# (Arsitektur modelnya pakai GlobalAveragePooling2D, jadi resolusi input
# BOLEH beda dari model lama -- bobot lama tetap bisa di-load, sudah
# diverifikasi tidak ada mismatch shape.)
IMG_SIZE = (128, 128)
CHANNELS = 3
BATCH_SIZE = 32
SEED = 42

# Nama kelas -> index. image_dataset_from_directory akan mengurutkan
# folder secara alfabetis, jadi fake=0, real=1.
CLASS_NAMES = ["fake", "real"]

# ---- Training ----
EPOCHS = 20
LEARNING_RATE = 3e-4  # lebih kecil dari training dari-nol (1e-3), karena warm-start
EARLY_STOPPING_PATIENCE = 4
FINE_TUNE_FROM_PRETRAINED = True  # set False untuk training dari nol (tanpa warm-start)


In [ ]:
%%writefile src/dataset.py
"""
dataset.py
Memuat gambar dari folder train/, valid/, test/ menjadi tf.data.Dataset.
Beda dari versi CIFAKE: dataset wajah ini sudah datang dengan split
train/valid/test dari pembuatnya, jadi tidak perlu potong validation_split
sendiri dari train/.
"""

import tensorflow as tf
from tensorflow.keras import layers

from . import config


def _prepare(ds: tf.data.Dataset, augment: bool, normalize: layers.Layer) -> tf.data.Dataset:
    ds = ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        augmentation = tf.keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.03),
            layers.RandomZoom(0.05),
            layers.RandomContrast(0.05),
        ])
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)

    return ds.prefetch(tf.data.AUTOTUNE)


def load_datasets():
    """
    Return: (train_ds, val_ds, test_ds)
    Label mengikuti urutan alfabetis nama folder -> fake=0, real=1
    (sudah didefinisikan juga di config.CLASS_NAMES).
    """
    normalize = layers.Rescaling(1.0 / 255)

    train_ds = tf.keras.utils.image_dataset_from_directory(
        config.TRAIN_DIR,
        labels="inferred",
        label_mode="binary",
        image_size=config.IMG_SIZE,
        batch_size=config.BATCH_SIZE,
        seed=config.SEED,
        shuffle=True,
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        config.VALID_DIR,
        labels="inferred",
        label_mode="binary",
        image_size=config.IMG_SIZE,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        config.TEST_DIR,
        labels="inferred",
        label_mode="binary",
        image_size=config.IMG_SIZE,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
    )

    train_ds = _prepare(train_ds, augment=True, normalize=normalize)
    val_ds = _prepare(val_ds, augment=False, normalize=normalize)
    test_ds = _prepare(test_ds, augment=False, normalize=normalize)

    return train_ds, val_ds, test_ds


In [ ]:
%%writefile src/model.py
"""
model.py
Arsitektur CNN ringan untuk klasifikasi biner REAL vs FAKE.

Desainnya terinspirasi dari pendekatan "lightweight CNN" yang dipakai
Lokner Ladjevic dkk. (2024, MDPI AI 5(3):1575-1593) untuk kasus yang
sama persis (deteksi gambar AI-generated di dataset CIFAKE): total
8 convolutional layer + 2 hidden (dense) layer. Catatan: paper tsb
tidak mempublikasikan tabel hyperparameter super detail (ukuran
kernel/filter tiap layer) di bagian yang bisa saya akses, jadi
implementasi berikut adalah desain umum & wajar yang mengikuti pola
tersebut (4 blok, tiap blok 2 conv layer), bukan replikasi 1:1 dari
kode aslinya. Jelaskan hal ini juga di laporan/skripsimu supaya jujur
soal mana yang "mengikuti ide paper" vs "hasil desain sendiri".

Total parameter model ini sekitar 300rb-600rb (tergantung filter),
jauh lebih kecil dari ResNet/VGG (puluhan juta parameter) -> cocok
disebut "algoritma ringan" di laporan kamu.
"""

from tensorflow.keras import layers, models

from . import config


def build_lightweight_cnn(input_shape=(32, 32, 3)) -> models.Model:
    inputs = layers.Input(shape=input_shape, name="image")

    x = inputs
    # 4 blok, masing-masing 2 conv layer -> total 8 conv layer
    filters_per_block = [32, 64, 128, 128]
    for i, f in enumerate(filters_per_block):
        x = layers.Conv2D(f, 3, padding="same", activation=None, name=f"block{i+1}_conv1")(x)
        x = layers.BatchNormalization(name=f"block{i+1}_bn1")(x)
        x = layers.Activation("relu", name=f"block{i+1}_act1")(x)

        x = layers.Conv2D(f, 3, padding="same", activation=None, name=f"block{i+1}_conv2")(x)
        x = layers.BatchNormalization(name=f"block{i+1}_bn2")(x)
        x = layers.Activation("relu", name=f"block{i+1}_act2")(x)

        # Jangan pooling lagi kalau feature map sudah kecil (image 32x32 -> setelah
        # 3x pooling jadi 4x4, blok ke-4 kita biarkan tanpa pooling tambahan)
        if i < 3:
            x = layers.MaxPooling2D(2, name=f"block{i+1}_pool")(x)
        x = layers.Dropout(0.25, name=f"block{i+1}_drop")(x)

    x = layers.GlobalAveragePooling2D(name="gap")(x)

    # 2 hidden (dense) layer
    x = layers.Dense(128, activation="relu", name="hidden1")(x)
    x = layers.Dropout(0.5, name="hidden1_drop")(x)
    x = layers.Dense(64, activation="relu", name="hidden2")(x)

    outputs = layers.Dense(1, activation="sigmoid", name="prediction")(x)

    model = models.Model(inputs, outputs, name="lightweight_cifake_cnn")
    return model


def compile_model(model: models.Model) -> models.Model:
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            "AUC",
            "Precision",
            "Recall",
        ],
    )
    return model


if __name__ == "__main__":
    m = build_lightweight_cnn(input_shape=(*config.IMG_SIZE, config.CHANNELS))
    m = compile_model(m)
    m.summary()


In [ ]:
%%writefile src/train.py
"""
train.py
Jalankan training dari root proyek dengan:

    python -m src.train

Beda dari versi CIFAKE: sebelum training, model di-warm-start dari bobot
model CIFAKE lama (models/cifake_cnn_best.keras) kalau file itu ada dan
config.FINE_TUNE_FROM_PRETRAINED=True. Arsitektur pakai GlobalAveragePooling2D
sehingga bobot tetap kompatibel walau resolusi input berbeda (32x32 -> 128x128).

Akan menghasilkan:
  - models/face_cnn_best.keras   (checkpoint terbaik berdasarkan val_loss)
  - models/face_cnn.keras        (model di epoch terakhir)
  - reports/training_history.png   (grafik akurasi & loss per epoch)
  - reports/training_history.csv   (angka mentahnya, buat lampiran laporan)
"""

import os
import csv

import tensorflow as tf
import matplotlib
matplotlib.use("Agg")  # supaya bisa jalan tanpa display (headless/server)
import matplotlib.pyplot as plt

from . import config
from .dataset import load_datasets
from .model import build_lightweight_cnn, compile_model


def plot_history(history, out_path):
    hist = history.history
    epochs_range = range(1, len(hist["loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(epochs_range, hist["loss"], label="train_loss")
    axes[0].plot(epochs_range, hist["val_loss"], label="val_loss")
    axes[0].set_title("Loss per Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs_range, hist["accuracy"], label="train_accuracy")
    axes[1].plot(epochs_range, hist["val_accuracy"], label="val_accuracy")
    axes[1].set_title("Akurasi per Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def save_history_csv(history, out_path):
    hist = history.history
    keys = list(hist.keys())
    n_epochs = len(hist[keys[0]])
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch"] + keys)
        for i in range(n_epochs):
            writer.writerow([i + 1] + [hist[k][i] for k in keys])


def main():
    os.makedirs(config.MODELS_DIR, exist_ok=True)
    os.makedirs(config.REPORTS_DIR, exist_ok=True)

    print(">> Memuat dataset ...")
    train_ds, val_ds, test_ds = load_datasets()

    print(">> Membangun model ...")
    model = build_lightweight_cnn(input_shape=(*config.IMG_SIZE, config.CHANNELS))
    model = compile_model(model)

    if config.FINE_TUNE_FROM_PRETRAINED and os.path.exists(config.PRETRAINED_CIFAKE_PATH):
        print(f">> Ditemukan model CIFAKE lama di {config.PRETRAINED_CIFAKE_PATH}")
        print(">> Warm-start: memuat bobotnya sebagai titik awal (transfer learning) ...")
        try:
            model.load_weights(config.PRETRAINED_CIFAKE_PATH)
            print(">> Berhasil! Training akan lanjut dari bobot CIFAKE, bukan dari nol.")
        except Exception as e:
            print(f">> Gagal load bobot lama ({e}); lanjut training dari nol.")
    else:
        print(">> Tidak ada model lama / FINE_TUNE_FROM_PRETRAINED=False -> training dari nol.")

    model.summary()

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            config.BEST_CHECKPOINT_PATH,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=config.EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    print(">> Mulai training ...")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=config.EPOCHS,
        callbacks=callbacks,
    )

    model.save(config.KERAS_MODEL_PATH)
    print(f">> Model tersimpan di: {config.KERAS_MODEL_PATH}")
    print(f">> Checkpoint terbaik di: {config.BEST_CHECKPOINT_PATH}")

    plot_history(history, os.path.join(config.REPORTS_DIR, "training_history.png"))
    save_history_csv(history, os.path.join(config.REPORTS_DIR, "training_history.csv"))
    print(">> Grafik & log training tersimpan di folder reports/")

    print(">> Evaluasi cepat di test set ...")
    results = model.evaluate(test_ds, return_dict=True)
    for k, v in results.items():
        print(f"   {k}: {v:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluate.py
"""
evaluate.py
Evaluasi model yang sudah dilatih terhadap test set: accuracy, precision,
recall, F1-score, confusion matrix (gambar), dan classification report
(teks). Semua hasil disimpan di folder reports/ supaya gampang ditempel
ke laporan/skripsi.

Jalankan dari root proyek:
    python -m src.evaluate
"""

import os

import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)

from . import config
from .dataset import load_datasets


def main(model_path: str = None):
    model_path = model_path or config.BEST_CHECKPOINT_PATH
    print(f">> Memuat model dari: {model_path}")
    model = tf.keras.models.load_model(model_path)

    print(">> Memuat test set ...")
    _, _, test_ds = load_datasets()

    y_true = []
    y_prob = []
    for batch_x, batch_y in test_ds:
        preds = model.predict(batch_x, verbose=0).flatten()
        y_prob.extend(preds.tolist())
        y_true.extend(batch_y.numpy().flatten().tolist())

    y_true = np.array(y_true, dtype=int)
    y_prob = np.array(y_prob, dtype=float)
    y_pred = (y_prob >= 0.5).astype(int)

    os.makedirs(config.REPORTS_DIR, exist_ok=True)

    report = classification_report(
        y_true, y_pred, target_names=config.CLASS_NAMES, digits=4
    )
    print(report)

    auc = roc_auc_score(y_true, y_prob)
    print(f"ROC-AUC: {auc:.4f}")

    with open(os.path.join(config.REPORTS_DIR, "classification_report.txt"), "w") as f:
        f.write(report)
        f.write(f"\nROC-AUC: {auc:.4f}\n")

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=config.CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion Matrix - CIFAKE Test Set")
    fig.tight_layout()
    fig.savefig(os.path.join(config.REPORTS_DIR, "confusion_matrix.png"), dpi=150)
    plt.close(fig)

    print(">> classification_report.txt & confusion_matrix.png tersimpan di reports/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/export_tflite.py
"""
export_tflite.py
Convert model .keras -> .tflite supaya bisa dipasang di aplikasi mobile
(Android/iOS) atau di alat yang resource-nya terbatas.

Menghasilkan 2 versi:
  1. cifake_cnn.tflite        -> float32 biasa, tanpa kompresi tambahan
  2. cifake_cnn_quant.tflite  -> dynamic range quantization (bobot di
     kompres ke int8), ukurannya jauh lebih kecil & inferensinya lebih
     cepat di HP, dengan penurunan akurasi yang biasanya kecil.

Jalankan dari root proyek:
    python -m src.export_tflite
"""

import os

import tensorflow as tf

from . import config


def convert(model_path: str, out_path: str, quantize: bool):
    model = tf.keras.models.load_model(model_path)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quantize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    with open(out_path, "wb") as f:
        f.write(tflite_model)

    size_kb = os.path.getsize(out_path) / 1024
    print(f">> Tersimpan: {out_path}  ({size_kb:.1f} KB)")
    return size_kb


def main():
    os.makedirs(config.MODELS_DIR, exist_ok=True)

    keras_size_kb = os.path.getsize(config.BEST_CHECKPOINT_PATH) / 1024
    print(f"Ukuran model Keras asli: {keras_size_kb:.1f} KB")

    plain_kb = convert(config.BEST_CHECKPOINT_PATH, config.TFLITE_MODEL_PATH, quantize=False)
    quant_kb = convert(config.BEST_CHECKPOINT_PATH, config.TFLITE_QUANT_MODEL_PATH, quantize=True)

    print("\nRingkasan ukuran model:")
    print(f"  Keras (.keras)            : {keras_size_kb:8.1f} KB")
    print(f"  TFLite float32 (.tflite)  : {plain_kb:8.1f} KB")
    print(f"  TFLite quantized (.tflite): {quant_kb:8.1f} KB")
    print("\nCatatan: setelah kuantisasi, cek ulang akurasinya di test set")
    print("(lihat predict.py / evaluate.py) sebelum dipasang ke app -")
    print("kadang ada sedikit penurunan akurasi yang perlu dicek dulu.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/predict.py
"""
predict.py
Coba model ke satu file gambar dari command line. Berguna buat demo cepat
sebelum bikin UI (Streamlit/mobile app).

Contoh pakai model Keras:
    python -m src.predict --image contoh.jpg

Contoh pakai model TFLite (yang nanti dipasang ke HP):
    python -m src.predict --image contoh.jpg --tflite models/cifake_cnn_quant.tflite
"""

import argparse

import numpy as np
import tensorflow as tf
from PIL import Image

from . import config


def load_image(path: str) -> np.ndarray:
    img = Image.open(path).convert("RGB").resize(config.IMG_SIZE)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr


def predict_keras(image_arr: np.ndarray, model_path: str) -> float:
    model = tf.keras.models.load_model(model_path)
    batch = np.expand_dims(image_arr, axis=0)
    prob = model.predict(batch, verbose=0)[0][0]
    return float(prob)


def predict_tflite(image_arr: np.ndarray, tflite_path: str) -> float:
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    batch = np.expand_dims(image_arr, axis=0).astype(input_details[0]["dtype"])
    interpreter.set_tensor(input_details[0]["index"], batch)
    interpreter.invoke()
    prob = interpreter.get_tensor(output_details[0]["index"])[0][0]
    return float(prob)


def interpret(prob_real: float) -> str:
    # Ingat: label index 1 = REAL, 0 = FAKE (lihat config.CLASS_NAMES)
    label = "REAL (foto asli)" if prob_real >= 0.5 else "FAKE (dibuat AI)"
    confidence = prob_real if prob_real >= 0.5 else 1 - prob_real
    return f"{label}  |  confidence: {confidence * 100:.1f}%"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", required=True, help="path ke file gambar")
    parser.add_argument("--model", default=config.BEST_CHECKPOINT_PATH,
                         help="path ke model .keras (dipakai kalau --tflite tidak diisi)")
    parser.add_argument("--tflite", default=None,
                         help="path ke model .tflite (opsional, override --model)")
    args = parser.parse_args()

    image_arr = load_image(args.image)

    if args.tflite:
        prob_real = predict_tflite(image_arr, args.tflite)
    else:
        prob_real = predict_keras(image_arr, args.model)

    print(interpret(prob_real))


if __name__ == "__main__":
    main()


---
### Langkah 3 — Upload model CIFAKE lama (untuk warm-start / transfer learning)

Upload file `cifake_cnn_best.keras` yang sudah kamu latih sebelumnya
(dari sesi Colab pertama). Kalau tidak diupload, training akan tetap
jalan tapi mulai dari bobot acak (dari nol) — lebih lambat konvergen.


In [ ]:
from google.colab import files
print("Upload file cifake_cnn_best.keras (opsional tapi disarankan):")
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, f"models/cifake_cnn_best.keras")
    print(f"Disimpan sebagai models/cifake_cnn_best.keras")


---
### Langkah 4 — Download dataset wajah dari Kaggle

1. Buka https://www.kaggle.com/settings/account → bagian **API** → **Create New Token**
   (kalau sudah pernah generate, tinggal salin username & key-nya).
2. Jalankan cell di bawah, upload `kaggle.json`.


In [ ]:
from google.colab import files
print("Upload file kaggle.json kamu di sini:")
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(list(uploaded.values())[0])
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("kaggle.json terpasang.")


In [ ]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p data --unzip
print("Download & extract selesai.")


In [ ]:
# Cek struktur folder hasil extract
import os
for root, dirs, fs in os.walk("data"):
    depth = root.replace("data", "").count(os.sep)
    if depth <= 3:
        n = len(fs)
        print(f"{root}  ->  {n} file" if n else root)


Dataset ini biasanya ter-extract dengan folder pembungkus seperti
`data/real_vs_fake/real-vs-fake/train/real`, dst. Cell berikut otomatis
mencari folder `train/valid/test` x `real/fake` di kedalaman berapapun,
lalu menatanya ulang jadi `data/train/real`, `data/train/fake`,
`data/valid/real`, `data/valid/fake`, `data/test/real`, `data/test/fake`
(sesuai yang diharapkan `src/config.py`). Aman dijalankan meski struktur
sudah pas -- tidak akan mengubah apapun kalau target sudah ada isinya.

In [ ]:
import shutil, glob

def find_and_flatten(split, cls):
    target = f"data/{split}/{cls}"
    if os.path.isdir(target) and len(os.listdir(target)) > 0:
        return
    candidates = [c for c in glob.glob(f"data/**/{cls}", recursive=True)
                  if split.lower() in c.lower() and os.path.isdir(c)]
    if candidates:
        os.makedirs(f"data/{split}", exist_ok=True)
        if os.path.abspath(candidates[0]) != os.path.abspath(target):
            shutil.move(candidates[0], target)
            print(f"Dipindahkan: {candidates[0]} -> {target}")
    else:
        print(f"⚠️  Tidak ketemu data/{split}/{cls}, cek manual dengan `!find data -type d`")

for split in ["train", "valid", "test"]:
    for cls in ["real", "fake"]:
        find_and_flatten(split, cls)

print("\nStruktur akhir:")
!find data -maxdepth 2 -type d | sort


---
### Langkah 5 — Training (dengan warm-start dari model CIFAKE)

Perhatikan log di awal: kalau `models/cifake_cnn_best.keras` ada, akan
muncul pesan **"Warm-start: memuat bobotnya..."** — tandanya transfer
learning berjalan, bukan training dari nol.


In [ ]:
!python -m src.train


In [ ]:
from IPython.display import Image, display
display(Image("reports/training_history.png"))


---
### Langkah 6 — Evaluasi

In [ ]:
!python -m src.evaluate


In [ ]:
from IPython.display import Image, display
display(Image("reports/confusion_matrix.png"))
with open("reports/classification_report.txt") as f:
    print(f.read())


---
### Langkah 7 — Export ke TensorFlow Lite

In [ ]:
!python -m src.export_tflite


---
### Langkah 8 — Download hasil ke laptopmu

Jangan lupa — begitu sesi Colab berakhir, semua file di sini hilang.


In [ ]:
import shutil
shutil.make_archive("hasil_training_face", "zip", ".", "models")
shutil.make_archive("hasil_laporan_face", "zip", ".", "reports")

from google.colab import files
files.download("hasil_training_face.zip")
files.download("hasil_laporan_face.zip")


**Opsional -- simpan ke Google Drive:**
```python
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/face-detector
!cp -r data models reports src /content/drive/MyDrive/face-detector/
```

---
### Catatan untuk laporan tugas akhir

- Model ini adalah **hasil transfer learning**: bobot awal dari CNN yang
  dilatih di CIFAKE (domain objek/hewan/kendaraan), lalu di-*fine-tune* ke
  domain wajah manusia asli vs StyleGAN. Ini bagian penting untuk
  dijelaskan di bab metodologi -- termasuk *kenapa* transfer learning
  dipakai (mengatasi temuan generalisasi dari eksperimen sebelumnya) dan
  *bagaimana* caranya bisa dilakukan meski resolusi input berbeda
  (arsitektur berbasis `GlobalAveragePooling2D` membuat bobot conv layer
  & dense layer tidak bergantung pada ukuran spasial input).
- Resolusi dinaikkan ke 128×128 (dari 32×32 di CIFAKE) karena wajah
  butuh detail lebih halus untuk membedakan artefak StyleGAN dari foto asli.
- Setelah training, uji lagi modelnya pakai foto-foto yang tadinya salah
  deteksi (selfie & foto grupmu) -- itu jadi bagian "validasi ulang" yang
  bagus untuk laporan: tunjukkan hasil SEBELUM (model CIFAKE) vs SESUDAH
  (model wajah ini) di foto yang sama.
